In [20]:
from pymongo import MongoClient
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()
connection_string = os.getenv("MONGO_URI")

client = MongoClient(connection_string)

# Connect to MongoDB
client = MongoClient(connection_string)

# Choose a database
db = client['cooking']

# Choose a collection
collection = db['recipes']

In [26]:
df = pd.read_csv('data/recipes_first_3k.csv')
df.sample()

,title,ingredients,directions,link,source,NER,site
1701,Date Bars,"[""2 blocks margarine"", ""2 c. flour"", ""1/2 c. s...","[""Mix together and bake 12 to 15 minutes at 32...",www.cookbooks.com/Recipe-Details.aspx?id=887344,Gathered,"[""flour"", ""margarine"", ""sugar""]",www.cookbooks.com


In [4]:
!ollama pull bge-m3

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠏ pulling manifest ⠏ pulling manifest ⠙ pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏  46 KB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 282 KB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 398 KB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 398 KB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 398 KB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 398 KB/1.2 GB                  pulling manifest 
p

#### Data ingestion

In [ ]:
import ollama
def embed(text):
    return ollama.embed(
    model='bge-m3',
    input=text,
    dimensions=1024
).embeddings[0]

#### Data ingestion

In [27]:
# Iterate over the DataFrame and insert into MongoDB
import ast


for _, row in df.iterrows():
    title_embedding = embed(row['title'])

    # Convert string representations of lists to actual lists
    try:
        ingredients_list = ast.literal_eval(row['ingredients']) if pd.notna(row['ingredients']) else []
        directions_list = ast.literal_eval(row['directions']) if pd.notna(row['directions']) else []
        ner_list = ast.literal_eval(row['NER']) if pd.notna(row['NER']) else []
    except Exception as e:
        print(f"Error parsing row {_}: {e}")
        ingredients_list, directions_list, ner_list = [], [], []

    doc = {
        "title": row['title'],
        "ingredients": ingredients_list,
        "directions": directions_list,
        "link": row['link'],
        "source": row['source'],
        "NER": ner_list,
        "site": row['site'],
        "embedding": title_embedding
    }

    collection.insert_one(doc)

print("All rows inserted with proper lists and embeddings!")

All rows inserted with proper lists and embeddings!
